# Phase 4.3：端到端验收、Demo 记录和最终交付

## 目标

把用户故事变成自动化验收：健康检查、检索引用、无 LLM fallback、错误输入、Demo 记录和项目测试。完成后，四个阶段的产物就真正组成一个可运行项目。

**本课交付：** `data/processed/phase4_acceptance_record.json`。

## Evidence Quest 任务卡：Phase 4.3：Demo Day 最终审判

**你的身份：** 项目发布负责人  
**案件背景：** 真实观众不会为你背诵术语。你要现场展示一次查询、引用、无证据 fallback 和自动化验收。

### 本关专业 Goal

完成从输入文件到可追溯回答的最终演示，并留下可复现记录。

### 你要交付的作品

**最终 Demo 记录 + 面试级项目展示**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：Evidence Quest 通关者  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 用户故事先于代码

> 作为需要查阅技术资料的实习生，我输入一个问题，希望得到可回到原文的证据；如果知识库没有足够信息，系统应该明确说不知道，而不是编造。

这个故事对应五类检查：服务可用、能搜索、有引用、无证据可解释、非法输入被拒绝。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase4.3'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase4.3
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 导入 FastAPI 测试客户端。
from fastapi.testclient import TestClient

# 导入应用工厂，创建独立测试应用。
from phase4_mini_rag_system.app import create_app

# 指定稳定的教学输入目录。
input_directory = ROOT / "phase1_doc_parser" / "examples" / "input"

# 创建应用并在启动阶段 ingest 文档。
app = create_app(input_directory)

# 创建可以发送 HTTP 请求的测试客户端。
client = TestClient(app)

# 输出应用已经准备好。
print("acceptance app ready")

acceptance app ready


## 2. 编写一组最小验收函数

函数的价值是避免把相同断言复制到很多地方，同时保留每个用户故事的名字。每条断言都应该在失败时告诉我们哪个合同被破坏。

In [4]:
# 定义健康检查验收函数。
def check_health():
    # 发送 health 请求。
    response = client.get("/health")

    # 断言 HTTP 层成功。
    assert response.status_code == 200, response.text

    # 读取 JSON 响应。
    payload = response.json()

    # 断言索引中有 Chunk。
    assert payload["chunks"] > 0

    # 返回结果供总记录使用。
    return payload

# 定义检索引用验收函数。
def check_search_citations():
    # 发送一条已知 Query。
    response = client.post("/search", json={"query": "Chunk overlap", "top_k": 3})

    # 断言请求成功。
    assert response.status_code == 200, response.text

    # 读取响应 JSON。
    payload = response.json()

    # 结果不能为空。
    assert payload["results"]

    # 检查引用的最小字段。
    required_fields = {"chunk_id", "source", "page", "score", "text"}
    assert required_fields <= payload["results"][0].keys()

    # 返回检索响应。
    return payload

In [5]:
# 定义聊天 fallback 验收函数。
def check_chat_fallback():
    # 发送聊天请求；默认环境没有开启 LLM。
    response = client.post("/chat", json={"query": "Chunk overlap", "top_k": 3})

    # 请求本身应该成功。
    assert response.status_code == 200, response.text

    # 读取聊天响应。
    payload = response.json()

    # 必须返回引用，不能只返回一句无来源答案。
    assert payload["citations"]

    # 记录模式，允许 evidence-only 或显式 fallback。
    assert payload["mode"] in {"evidence-only", "evidence-only-fallback", "llm"}

    # 返回聊天响应供 Demo 记录。
    return payload

# 定义非法 Query 验收函数。
def check_invalid_query():
    # 发送空 Query。
    response = client.post("/search", json={"query": ""})

    # 空 Query 应该是 422，而不是服务器 500。
    assert response.status_code == 422

    # 返回状态码供记录。
    return response.status_code

## 3. 执行完整验收并保存 Demo

现在才运行刚才定义的检查。把响应中的索引版本、模式和 citation 摘要保存下来，形成一次可回放的项目证据。

In [6]:
# 执行健康检查并保存结果。
health_payload = check_health()

# 执行检索引用检查并保存结果。
search_payload = check_search_citations()

# 执行聊天 fallback 检查并保存结果。
chat_payload = check_chat_fallback()

# 执行非法输入检查并保存状态码。
invalid_status = check_invalid_query()

# 提取引用中的稳定字段，避免记录过大的正文。
citation_summary = []
for citation in chat_payload["citations"]:
    # 保存用户回溯原文所需的字段。
    citation_summary.append({"chunk_id": citation["chunk_id"], "source": citation["source"], "page": citation["page"], "score": citation["score"]})

# 组合一次端到端验收记录。
acceptance_record = {"health": health_payload, "search_trace_id": search_payload["trace_id"], "chat_trace_id": chat_payload["trace_id"], "chat_mode": chat_payload["mode"], "invalid_query_status": invalid_status, "citations": citation_summary}

# 指定最终验收记录路径。
acceptance_path = ROOT / "data" / "processed" / "phase4_acceptance_record.json"

# 保存验收记录。
acceptance_path.write_text(json.dumps(acceptance_record, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印交付路径和聊天模式。
print("已生成:", acceptance_path)
print("chat mode:", chat_payload["mode"])

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\phase4_acceptance_record.json
chat mode: evidence-only


## 4. 运行项目测试：Notebook 验收之后还要有长期保护

Notebook 断言适合边学边反馈，`pytest` 测试适合以后修改代码时持续保护行为。这里用当前环境运行项目测试，并检查返回码。

In [7]:
# 导入 subprocess，用它调用项目测试命令。
import subprocess

# 用当前 Python 解释器运行 pytest，避免切换到错误环境。
test_process = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, capture_output=True, text=True)

# 打印 pytest 的最后几行，快速查看测试结果。
print(test_process.stdout[-1000:])

# 如果测试失败，打印标准错误帮助定位。
if test_process.returncode != 0:
    print(test_process.stderr)

# 所有项目测试必须通过，最终项目才算可交付。
assert test_process.returncode == 0

.....s......                                                             [100%]
============================== warnings summary ===============================
F:\anaconda\miniconda3\Lib\site-packages\starlette\formparsers.py:12
  F:\anaconda\miniconda3\Lib\site-packages\starlette\formparsers.py:12: PendingDeprecationWarning: Please use `import python_multipart` instead.
    import multipart

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
11 passed, 1 skipped, 1 warning in 0.63s



## 最终 0→1 项目验收

你现在可以从根目录启动真实服务：

```powershell
conda activate 'F:\anaconda\miniconda3\envs\ai-rag-internship'
python -m phase4_mini_rag_system
```

然后打开 `http://127.0.0.1:8000/`。最终项目已经具备：

```text
文件解析 -> 中文分块 -> 稳定 Chunk ID -> BM25 检索
-> 质量/性能实验 -> FastAPI -> evidence-only 引用回答
```

如果要继续升级，下一条独立实验才是 BGE-M3 Dense、Faiss HNSW、ONNX/INT8 和真实 LLM；这些升级必须沿用已有 qrels、benchmark 和引用合同。

## Phase 4 最终闸门

- [ ] 能从用户故事解释每一条 API 断言。
- [ ] `/health`、`/search`、`/chat` 和错误输入均已验证。
- [ ] 答案包含 citations，且无证据时不会编造。
- [ ] `phase4_acceptance_record.json` 已生成。
- [ ] pytest 全部通过。

至此，学习不是停在教程代码，而是完成了一套可运行、可解释、可测试的完整项目。

## Boss Challenge：用一个真实观众会问的问题跑完整 Demo，并记录 citation 的 chunk_id、source、page。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [8]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [9]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase4_acceptance_record.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase4_acceptance_record.json']
